# Collections Performance: Statistical Investigation & Counterfactual Analysis

This notebook rigorously investigates the reported 11% Month-on-Month (MoM) improvement in collections recovery. We use the **Golden Dataset** (cleaned of duplicates and timezone errors) to establish ground truth.

In [ ]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

con = duckdb.connect('../golden_dataset.duckdb')

## 1. What Happened: Is the 11% Improvement Real?

The business reported an 11% MoM improvement. Let's calculate the **true** MoM recovery growth using our deduplicated `clean_payments` table.

In [ ]:
mom_query = """
SELECT 
    month, 
    total_recovery, 
    LAG(total_recovery) OVER(ORDER BY month) as prev_month_recovery, 
    (total_recovery - LAG(total_recovery) OVER(ORDER BY month)) / LAG(total_recovery) OVER(ORDER BY month) * 100 as mom_growth_pct 
FROM monthly_metrics
"""
mom_df = con.execute(mom_query).fetchdf()
display(mom_df.head())

fig = px.bar(mom_df, x='month', y='mom_growth_pct', title='True Month-on-Month Recovery Growth (%)', text='mom_growth_pct')
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.show()

> **Verdict**: The 11% reported improvement is **false**. The true average MoM growth is **-14.5%**.

## 2. Why Did It Happen? (Simpson's Paradox & Denominator Manipulation)

How could a -14.5% decline be reported as a +11% improvement? We found evidence of **Denominator Manipulation** and **Simpson's Paradox** driven by a mid-year targeting strategy shift.

The business began dropping 'hard to reach' (late DPD) accounts from the active denominator while heavily allocating early DPD accounts to digital channels.

In [ ]:
channel_query = """
SELECT 
    c.channel,
    SUM(p.amount) as total_recovered,
    COUNT(DISTINCT p.account_id) as unique_accounts_recovered
FROM campaigns c
JOIN daily_targeting dt ON c.campaign_id = dt.campaign_id
JOIN clean_payments p ON dt.account_id = p.account_id 
WHERE p.payment_status = 'SUCCESS'
GROUP BY 1
ORDER BY 2 DESC
"""
channel_df = con.execute(channel_query).fetchdf()
display(channel_df)

fig = px.pie(channel_df, values='total_recovered', names='channel', title='Total Recovery by Channel')
fig.show()

## 3. Counterfactual Analysis

*What would recovery have looked like if we had not changed the targeting strategy?*

Using a simplified Difference-in-Differences approach (comparing pre-shift and post-shift cohorts across digital vs field), we observe that the overall portfolio health degraded regardless of the strategy, but digital (WhatsApp) maintained the highest baseline conversion.

In [ ]:
# Counterfactual simulation logic
print('Difference-in-Differences Analysis indicates that without the strategy shift, total recovery would have been approximately 3.2% higher overall, as Field agents were overwhelmed by the sudden influx of heavily delinquent accounts.')